In [1]:
import dolfin as dl
import ufl
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import sys
import os
from utils.hippylib import *

import logging
logging.getLogger('FFC').setLevel(logging.WARNING)
logging.getLogger('UFL').setLevel(logging.WARNING)
dl.set_log_active(False)

## 2. Construct the velocity field

In [4]:
def v_boundary(x,on_boundary):
    return on_boundary

def q_boundary(x,on_boundary):
    return x[0] < dl.DOLFIN_EPS and x[1] < dl.DOLFIN_EPS
        
def computeVelocityField(mesh):
    Xh = dl.VectorFunctionSpace(mesh,'Lagrange', 2)
    Wh = dl.FunctionSpace(mesh, 'Lagrange', 1)
    mixed_element = ufl.MixedElement([Xh.ufl_element(), Wh.ufl_element()])
    XW = dl.FunctionSpace(mesh, mixed_element)

    Re = dl.Constant(1e2)
    
    g = dl.Expression(('0.0','(x[0] < 1e-14) - (x[0] > 1 - 1e-14)'), degree=1)
    bc1 = dl.DirichletBC(XW.sub(0), g, v_boundary)
    bc2 = dl.DirichletBC(XW.sub(1), dl.Constant(0), q_boundary, 'pointwise')
    bcs = [bc1, bc2]
    
    vq = dl.Function(XW)
    (v,q) = ufl.split(vq)
    (v_test, q_test) = dl.TestFunctions (XW)
    
    def strain(v):
        return ufl.sym(ufl.grad(v))
    
    F = ( (2./Re)*ufl.inner(strain(v),strain(v_test))+ ufl.inner (ufl.nabla_grad(v)*v, v_test)
           - (q * ufl.div(v_test)) + ( ufl.div(v) * q_test) ) * ufl.dx
           
    dl.solve(F == 0, vq, bcs, solver_parameters={"newton_solver":
                                         {"relative_tolerance":1e-4, "maximum_iterations":100}})
    
    plt.figure(figsize=(15,5))
    vh = dl.project(v,Xh)
    qh = dl.project(q,Wh)
    nb.plot(nb.coarsen_v(vh), subplot_loc=121,mytitle="Velocity")
    nb.plot(qh, subplot_loc=122,mytitle="Pressure")
    plt.show()
        
    return v

## 3. Set up the mesh and finite element spaces

In [5]:
mesh = dl.refine( dl.Mesh("ad_20.xml") )
wind_velocity = computeVelocityField(mesh)
Vh = dl.FunctionSpace(mesh, "Lagrange", 1)
print( "Number of dofs: {0}".format( Vh.dim() ))

In [42]:
u = dl.Function(Vh).vector()
v = dl.Function(Vh).vector()
k = 10
p = 20
Omega = MultiVector(u, k+p)
# print all atrributes of Omega
u_2 = Omega[0].size()
print(u_2)

In [6]:
ic_expr = dl.Expression(
    'std::min(2.,std::exp(-20*(std::pow(x[0]-0.35,2) +  std::pow(x[1]-0.7,2))))',
    element=Vh.ufl_element())
true_initial_condition = dl.interpolate(ic_expr, Vh).vector()

In [6]:
from hippylib.modeling import prior
import fenics as fn
from scipy.linalg import sqrtm
from petsc4py import PETSc

class GaussianPrior:
    def __init__(self, Vh, covariance, mean=None):
        """
        Constructor

        Inputs:
        - :code:`Vh`:             Finite element space on which the prior is
                                  defined. Must be the Real space with one global 
                                  degree of freedom
        - :code:`covariance`:     The covariance of the prior. Must be a
                                  :code:`numpy.ndarray` of appropriate size
        - :code:`mean`(optional): Mean of the prior distribution. Must be of
                                  type `dolfin.Vector()`
        """

        self.Vh = Vh
        rel_tol = 1e-12
        max_iter = 1000
    
        if Vh.dim() != covariance.shape[0] or Vh.dim() != covariance.shape[1]:
            raise ValueError("Covariance incompatible with Finite Element space")

        if not np.issubdtype(covariance.dtype, np.floating):
            raise TypeError("Covariance matrix must be a float array")

        
        trial = fn.TrialFunction(Vh)
        test  = fn.TestFunction(Vh)
        
        varfM = fn.inner(trial, test) * fn.dx
         
        self.M = fn.assemble(varfM)
        covariance_discrete = self.M.array() @ covariance
        # Define R:
        temp_R = PETSc.Mat()
        temp_R.createDense(covariance.shape, array=covariance_discrete)
        temp_R.assemble()
        R = fn.PETScMatrix(temp_R)
        self.R = fn.as_backend_type(R)
        
        # Define sqrtR:
        sqrt_covariance = sqrtm(covariance)
        temp_sqrtR = PETSc.Mat()
        temp_sqrtR.createDense(sqrt_covariance.shape, array=sqrt_covariance)
        temp_sqrtR.assemble()
        self.sqrtR = fn.PETScMatrix(temp_sqrtR)
        
        self.Rsolver = fn.PETScKrylovSolver("cg")
        self.Rsolver.set_operator(self.M)
        self.Rsolver.parameters["maximum_iterations"] = max_iter
        self.Rsolver.parameters["relative_tolerance"] = rel_tol
        self.Rsolver.parameters["error_on_nonconvergence"] = True
        self.Rsolver.parameters["nonzero_initial_guess"] = False
        
        self.Msolver = PETScKrylovSolver(self.Vh.mesh().mpi_comm(), "cg", "jacobi")
        self.Msolver.set_operator(self.M)
        self.Msolver.parameters["maximum_iterations"] = max_iter
        self.Msolver.parameters["relative_tolerance"] = rel_tol
        self.Msolver.parameters["error_on_nonconvergence"] = True
        self.Msolver.parameters["nonzero_initial_guess"] = False
        
        if mean:
            self.mean = mean
        else:
            tmp = fn.Vector()
            self.M.init_vector(tmp, 0)
            tmp.zero()
            self.mean = tmp
        
    def init_vector(self, x, dim):
        """
        Inizialize a vector :code:`x` to be compatible with the 
        range/domain of :math:`R`.

        If :code:`dim == "noise"` inizialize :code:`x` to be compatible 
        with the size of white noise used for sampling.
        """

        if dim == "noise":
            # self.sqrtRinv.init_vector(x, 1)
            self.sqrtR.init_vector(x, 1)
        else:
            # self.sqrtRinv.init_vector(x, dim)
            self.sqrtR.init_vector(x, dim)

    def sample(self, noise, s, add_mean=True):
        """
        Given :code:`noise` :math:`\\sim \\mathcal{N}(0, I)` compute a 
        sample :code:`s` from the prior.

        If :code:`add_mean == True` add the prior mean value to :code:`s`.
        """
       
        self.sqrtRinv.mult(noise, s)

        if add_mean:
            s.axpy(1.0, self.mean)

In [7]:
t_init         = 0.
t_final        = 4.
t_1            = 1.
dt             = .1
observation_dt = .1
    
simulation_times = np.arange(t_init, t_final+.5*dt, dt)
observation_times = np.arange(t_1, t_final+.5*dt, observation_dt)
    
targets = np.loadtxt('targets.txt')
print ("Number of observation points: {0}".format(targets.shape[0]) )
misfit = SpaceTimePointwiseStateObservation(Vh, observation_times, targets)

In [10]:
u = fn.Function(Vh).vector()
k = 80
p = 20
Omega = MultiVector(u, k+p)
print(Omega.shape)

In [7]:
class GaussianKernel:
    def __init__(self, sigma, correlation_length):
        self.sigma = sigma
        self.corr_length = correlation_length
        self.value = lambda x, y: self.sigma ** 2 * np.exp(-np.sqrt(((x[0] - y[0]) ** 2 + (x[1] - y[1]) ** 2)) / (2 * self.corr_length))

# Kernel function:
def K_matrix_2D(kernel, X, Y, indexing='xy'):
    value = kernel.value
    # Input: A kernel k and list of coordinates V.
    # Output: K: K_ij = k(X_i, Y_j), grad_K: grad_K_ij = grad_X k(X_i, Y_j).
    X_mesh_1, Y_mesh_1 = np.meshgrid(X[:, 0], Y[:, 0], indexing=indexing)
    X_mesh_2, Y_mesh_2 = np.meshgrid(X[:, 1], Y[:, 1], indexing=indexing)
    # Compute function:
    value_F = value([X_mesh_1, X_mesh_2], [Y_mesh_1, Y_mesh_2])
    return value_F

In [8]:
sigma = 0.2
correlation_length = 0.05
kernel = GaussianKernel(sigma, correlation_length)
XX = Vh.tabulate_dof_coordinates()
K = K_matrix_2D(kernel, XX, XX)
K[np.where(np.abs(K) < 1e-14)] = 0
covariance = sigma ** 2 * np.eye(Vh.dim())

In [9]:
prior = GaussianPrior(Vh, covariance)
# prior = BiLaplacianPrior(Vh, 8., 1.)
prior_2 = BiLaplacianPrior(Vh, 10., 1.)
problem = TimeDependentAD(mesh, [Vh,Vh,Vh], prior, misfit, simulation_times, wind_velocity, True)

## 5. Generate the synthetic observations

In [10]:
rel_noise = 0.01
utrue = problem.generate_vector(STATE)
x = [utrue, true_initial_condition, None]
problem.solveFwd(x[STATE], x)
misfit.observe(x, misfit.d)
MAX = misfit.d.norm("linf", "linf")
noise_std_dev = rel_noise * MAX
parRandom.normal_perturb(noise_std_dev,misfit.d)
misfit.noise_variance = noise_std_dev*noise_std_dev

In [11]:
[u,m,p] = problem.generate_vector()
problem.solveFwd(u, [u,m,p])
problem.solveAdj(p, [u,m,p])
mg = problem.generate_vector(PARAMETER)
grad_norm = problem.evalGradientParameter([u,m,p], mg)
print( "(g,g) = ", grad_norm)

In [12]:
H = ReducedHessian(problem, misfit_only=True) 
k = 80
p = 20
print( "Single Pass Algorithm. Requested eigenvectors: {0}; Oversampling {1}.".format(k,p) )
Omega = MultiVector(x[PARAMETER], k+p)
parRandom.normal(1., Omega)
lmbda, V = singlePassG(H, prior.R, prior.Rsolver, Omega, k)
posterior = GaussianLRPosterior(prior, lmbda, V)

In [17]:
Omega = MultiVector(x[PARAMETER], k+p)
print(Omega)

In [13]:
H.misfit_only = False
        
solver = CGSolverSteihaug()
solver.set_operator(H)
trial, test = dl.TrialFunction(Vh), dl.TestFunction(Vh)
solver.set_preconditioner(prior_2.Rsolver)
solver.parameters["print_level"] = 1
solver.parameters["rel_tolerance"] = 1e-6
solver.solve(m, -mg)
problem.solveFwd(u, [u,m,p])
 
total_cost, reg_cost, misfit_cost = problem.cost([u,m,p])
print( "Total cost {0:5g}; Reg Cost {1:5g}; Misfit {2:5g}".format(total_cost, reg_cost, misfit_cost) )
    
posterior.mean = m

In [14]:
plt.figure(figsize=(7.5,5))
nb.plot(dl.Function(Vh, m), mytitle="Initial Condition", vmin=0, vmax=2)

plt.show()

In [15]:
plt.figure(figsize=(7.5,5))
nb.plot(dl.Function(Vh, true_initial_condition), mytitle="Initial Condition", vmin=0, vmax=2)
plt.show()